# 02 — Build RAG Pipeline

Ecommerce KPI RAG Capstone — LLM Zoomcamp 2026

This notebook turns the merged transaction data into a set of retrievable KPI documents,
builds a text search index (minsearch) and a vector search index (sentence-transformers),
and wires both up to Groq for answer generation.

**Retrieval flow:** question -> retrieve top-k KPI documents -> build prompt -> Groq -> answer

## 1. Load merged data

In [30]:
import pandas as pd

df = pd.read_csv("../data/merged_transactions.csv", parse_dates=["timestamp"])
campaigns = pd.read_csv("../data/campaigns.csv")
df = df.dropna(subset=["gross_revenue"])  # exclude the 10,449 rows with missing revenue (see Notebook 01)
print(df.shape)

(92678, 21)


## 2. Build KPI documents

Rather than indexing raw transactions (too granular to answer questions like *"what was total
revenue last quarter?"*), we pre-aggregate the data into KPI summary documents — one per
category, country, loyalty tier, month, and campaign channel. Each document is a short text
blurb the retriever can match against a question, with the underlying numbers included so the
LLM can quote them directly.

In [31]:
documents = []
doc_id = 0

def add_doc(text, doc_type, **meta):
    global doc_id
    documents.append({"id": doc_id, "type": doc_type, "text": text, **meta})
    doc_id += 1

# --- Category summaries ---
for cat, g in df.groupby("category"):
    add_doc(
        f"Category: {cat}. Total revenue: ${g['gross_revenue'].sum():,.2f}. "
        f"Orders: {len(g):,}. Average order value: ${g['gross_revenue'].mean():,.2f}. "
        f"Refund rate: {g['refund_flag'].mean():.2%}.",
        doc_type="category", category=cat
    )

# --- Country summaries ---
for country, g in df.groupby("country"):
    add_doc(
        f"Country: {country}. Total revenue: ${g['gross_revenue'].sum():,.2f}. "
        f"Orders: {len(g):,}. Average order value: ${g['gross_revenue'].mean():,.2f}. "
        f"Unique customers: {g['customer_id'].nunique():,}.",
        doc_type="country", country=country
    )

# --- Loyalty tier summaries ---
for tier, g in df.groupby("loyalty_tier"):
    add_doc(
        f"Loyalty tier: {tier}. Total revenue: ${g['gross_revenue'].sum():,.2f}. "
        f"Orders: {len(g):,}. Average order value: ${g['gross_revenue'].mean():,.2f}.",
        doc_type="loyalty_tier", loyalty_tier=tier
    )

# --- Monthly summaries ---
df["month"] = df["timestamp"].dt.to_period("M").astype(str)
for month, g in df.groupby("month"):
    add_doc(
        f"Month: {month}. Total revenue: ${g['gross_revenue'].sum():,.2f}. Orders: {len(g):,}.",
        doc_type="month", month=month
    )

# --- Campaign channel summaries ---
df_campaigns = df[df["campaign_id"] > 0].merge(campaigns, on="campaign_id", how="left")
for channel, g in df_campaigns.groupby("channel"):
    add_doc(
        f"Marketing channel: {channel}. Revenue attributed: ${g['gross_revenue'].sum():,.2f}. "
        f"Orders: {len(g):,}.",
        doc_type="channel", channel=channel
    )

# --- Overall KPI summary ---
add_doc(
    f"Overall summary: Total revenue across all orders is ${df['gross_revenue'].sum():,.2f} "
    f"from {len(df):,} orders. Average order value is ${df['gross_revenue'].mean():,.2f}. "
    f"Refund rate is {df['refund_flag'].mean():.2%}.",
    doc_type="overall"
)

print(f"Built {len(documents)} KPI documents")
documents[:3]

Built 59 KPI documents


[{'id': 0,
  'type': 'category',
  'text': 'Category: Beauty. Total revenue: $369,671.88. Orders: 9,224. Average order value: $40.08. Refund rate: 3.10%.',
  'category': 'Beauty'},
 {'id': 1,
  'type': 'category',
  'text': 'Category: Electronics. Total revenue: $3,452,007.19. Orders: 21,095. Average order value: $163.64. Refund rate: 2.97%.',
  'category': 'Electronics'},
 {'id': 2,
  'type': 'category',
  'text': 'Category: Fashion. Total revenue: $1,298,075.77. Orders: 19,339. Average order value: $67.12. Refund rate: 2.79%.',
  'category': 'Fashion'}]

In [32]:
import json

with open("../data/documents.json", "w") as f:
    json.dump(documents, f, indent=2)

print("Saved data/documents.json:", len(documents), "documents")

Saved data/documents.json: 59 documents


## 3. Text search index (minsearch)

`minsearch` is the lightweight keyword-search library used throughout the LLM Zoomcamp course.
Install it with `pip install minsearch` (already in `requirements.txt`).

In [33]:
from minsearch import Index

text_index = Index(
    text_fields=["text"],
    keyword_fields=["type", "category", "country", "loyalty_tier", "month", "channel"],
)
text_index.fit(documents)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

# quick smoke test
text_search("revenue in Electronics category")

[{'id': 1,
  'type': 'category',
  'text': 'Category: Electronics. Total revenue: $3,452,007.19. Orders: 21,095. Average order value: $163.64. Refund rate: 2.97%.',
  'category': 'Electronics'},
 {'id': 10,
  'type': 'country',
  'text': 'Country: IN. Total revenue: $1,679,708.93. Orders: 18,636. Average order value: $90.13. Unique customers: 12,042.',
  'country': 'IN'},
 {'id': 0,
  'type': 'category',
  'text': 'Category: Beauty. Total revenue: $369,671.88. Orders: 9,224. Average order value: $40.08. Refund rate: 3.10%.',
  'category': 'Beauty'},
 {'id': 5,
  'type': 'category',
  'text': 'Category: Sports. Total revenue: $965,609.13. Orders: 10,168. Average order value: $94.97. Refund rate: 3.24%.',
  'category': 'Sports'},
 {'id': 4,
  'type': 'category',
  'text': 'Category: Home. Total revenue: $1,996,432.94. Orders: 18,421. Average order value: $108.38. Refund rate: 2.75%.',
  'category': 'Home'}]

## 4. Vector search index (sentence-transformers)

In [34]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_texts = [d["text"] for d in documents]
doc_embeddings = model.encode(doc_texts, show_progress_bar=True)

def vector_search(query, num_results=5):
    query_vec = model.encode([query])[0]
    sims = doc_embeddings @ query_vec / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_vec) + 1e-9
    )
    top_idx = np.argsort(sims)[::-1][:num_results]
    return [documents[i] for i in top_idx]

# quick smoke test
vector_search("revenue in Electronics category")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[{'id': 1,
  'type': 'category',
  'text': 'Category: Electronics. Total revenue: $3,452,007.19. Orders: 21,095. Average order value: $163.64. Refund rate: 2.97%.',
  'category': 'Electronics'},
 {'id': 54,
  'type': 'channel',
  'text': 'Marketing channel: Display. Revenue attributed: $1,211,416.38. Orders: 13,437.',
  'channel': 'Display'},
 {'id': 10,
  'type': 'country',
  'text': 'Country: IN. Total revenue: $1,679,708.93. Orders: 18,636. Average order value: $90.13. Unique customers: 12,042.',
  'country': 'IN'},
 {'id': 12,
  'type': 'country',
  'text': 'Country: US. Total revenue: $2,950,087.57. Orders: 32,310. Average order value: $91.31. Unique customers: 20,914.',
  'country': 'US'},
 {'id': 22,
  'type': 'month',
  'text': 'Month: 2021-06. Total revenue: $219,639.13. Orders: 2,406.',
  'month': '2021-06'}]

## 5. Groq generation

In [35]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

PROMPT_TEMPLATE = """You are an ecommerce analytics assistant. Answer the QUESTION using only
the numbers in CONTEXT. If the context doesn't contain the answer, say so — don't guess.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

def build_context(results):
    return "\n".join(f"- {r['text']}" for r in results)

def rag_answer(question, search_fn=text_search, num_results=6):
    results = search_fn(question, num_results=num_results)
    context = build_context(results)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="low",
    )
    return response.choices[0].message.content, results


In [36]:

answer, sources = rag_answer("Which category generated the most revenue?")
print(answer)

Electronics.


In [37]:
answer, sources = rag_answer("Which category generated the most revenue?", num_results=6)
print(answer)
print()
for s in sources:
    print(s["text"])


Electronics generated the most revenue.

Category: Beauty. Total revenue: $369,671.88. Orders: 9,224. Average order value: $40.08. Refund rate: 3.10%.
Category: Sports. Total revenue: $965,609.13. Orders: 10,168. Average order value: $94.97. Refund rate: 3.24%.
Category: Home. Total revenue: $1,996,432.94. Orders: 18,421. Average order value: $108.38. Refund rate: 2.75%.
Category: Fashion. Total revenue: $1,298,075.77. Orders: 19,339. Average order value: $67.12. Refund rate: 2.79%.
Category: Grocery. Total revenue: $292,169.45. Orders: 14,431. Average order value: $20.25. Refund rate: 2.88%.
Category: Electronics. Total revenue: $3,452,007.19. Orders: 21,095. Average order value: $163.64. Refund rate: 2.97%.


## Summary

- Built **59 KPI documents** (6 category + 7 country + 4 loyalty tier + 36 monthly + 5 channel
  + 1 overall) — saved to `data/documents.json`
- Set up both a **text search index** (minsearch) and a **vector search index**
  (sentence-transformers, `all-MiniLM-L6-v2`) over the same document set
- Wired retrieval into a Groq-based generation function (`rag_answer`)

Notebook 03 evaluates the two retrieval approaches against each other; Notebook 04 compares
two different prompt strategies for generation.